# Zell: 50M-token validation run (Kaggle 2xT4)

Goal of this notebook: prove the pipeline end to end on a small budget before the full run.
Build the mixed blend (general text + code + math + chat + synthetic tool-call data), continued-pretrain
Qwen2.5-0.5B-Instruct on ~50M tokens, benchmark WikiText-103 strided perplexity against the baselines,
and sample generations to confirm it produces real content.

Two-notebook operating model (carried from BrainFormer): the blend build needs **Internet ON**; the heavy
training pass can run **Internet OFF** once the .bin/meta.json are on disk. For this small validation run
it is fine to do both here with internet on.

Baseline of comparison: Pythia-410M, WikiText-103 strided ppl **17.19**.

## 0. Setup
The next cell clones the public repo into `ZELL` (change `REPO` if you forked it). Synthetic data, if
generated with `synth/generate.py`, should be uploaded as a Kaggle dataset and matched by `SYNTH_GLOB`.

In [ ]:
import os
REPO = "https://github.com/PlangoDev/Zell.git"
ZELL = "/kaggle/working/Zell"          # repo checkout location
WORK = "/kaggle/working"
SYNTH_GLOB = "/kaggle/input/zell-synth/*.jsonl"   # set to "" to skip synth source
POOL_TOKENS = 50_000_000               # validation pool
TRAIN_TOKENS = 50_000_000              # tokens the trainer consumes
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
os.environ.setdefault("HF_DATASETS_CACHE", "/tmp/hfcache")
print(REPO, ZELL, WORK)

In [ ]:
# clone (or refresh) the code, then install deps. --depth 1 keeps it light.
!rm -rf {ZELL} && git clone --depth 1 {REPO} {ZELL}
!pip -q install -r {ZELL}/requirements.txt
# Liger fused kernels (fused linear-CE) are the main speed lever; tolerate failure
# on GPUs where Triton can't build (training auto-falls back to the standard path).
!pip -q install liger-kernel 2>/dev/null && echo "liger-kernel installed" || echo "liger-kernel unavailable; standard path"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Build data (Internet ON)
WikiText-103 test bin (the benchmark) + the mixed blend memmap. The blend ingests the synthetic
JSONL as one weighted source if `SYNTH_GLOB` matches files.

In [ ]:
# WikiText-103 test set in the Qwen tokenizer (headline benchmark)
!python {ZELL}/tools/build_data.py --model {MODEL_ID} --out-dir {WORK} --test-only

In [ ]:
# Mixed pretrain blend -> tokens.bin + meta.json
!python {ZELL}/tools/build_blend.py --model {MODEL_ID} --pool-tokens {POOL_TOKENS} \
    --train-tokens {TRAIN_TOKENS} --out-dir {WORK} --synth-glob "{SYNTH_GLOB}"
import json; print(json.load(open(f'{WORK}/meta.json')))

## 2. Continued-pretrain on the blend (DDP across both T4s)
fp16 is the fast default on Turing T4 (no bf16 tensor cores). Checkpoints land in `zell-core/`.

In [ ]:
# Batch/accum/checkpointing auto-size from whether Liger loaded (see train.py).
# Liger ON -> batch 8, no gradient checkpointing -> big throughput gain on 2xT4.
# If this OOMs (Liger failed to build), add --per-device-batch 2.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True accelerate launch \
    --multi_gpu --num_processes 2 --num_machines 1 --dynamo_backend no --mixed_precision fp16 \
    {ZELL}/train/train.py --meta {WORK}/meta.json --train-tokens {TRAIN_TOKENS} \
    --out-dir {WORK}/zell-core --precision fp16 --seq-len 1024

## 3. Benchmark: WikiText-103 strided ppl vs the baselines
Zell-vs-Qwen is an exact comparison (same tokenizer); Pythia/TinyLlama/SmolLM2 are indicative
(per-token ppl is tokenizer-dependent), shown for context against the 17.19 reference.

In [ ]:
TEST_BIN = f"{WORK}/wikitext103_test_{MODEL_ID.replace('/', '_')}.bin"
MODELS = ",".join([
    "EleutherAI/pythia-410m",
    "Qwen/Qwen2.5-0.5B-Instruct",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    f"{WORK}/zell-core",
])
!python {ZELL}/tools/bench_ppl.py --test-bin {TEST_BIN} --dtype uint32 --models "{MODELS}" --out {WORK}/scoreboard.json
import json; print(json.dumps(json.load(open(f'{WORK}/scoreboard.json')), indent=2))

## 4. Generate: does it produce real content?
The trainer already prints samples at the end; this cell re-loads the saved core for ad-hoc prompts,
including a tool-call probe.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
core = f"{WORK}/zell-core"
tok = AutoTokenizer.from_pretrained(core)
model = AutoModelForCausalLM.from_pretrained(core, torch_dtype=torch.float16).cuda().eval()
for p in ["Explain photosynthesis to a 10-year-old.",
          "Write a Python function to check if a string is a palindrome.",
          "What's the weather in Paris? Use a tool if you need one."]:
    text = tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=160, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tok.eos_token_id)
    print("\nUSER:", p)
    print("ZELL:", tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip())